# DeBERTa-V3 VAD Regression on EmoBank

Fine-tunes `microsoft/deberta-v3-base` (loaded from the Hugging Face Hub) to predict **Valence, Arousal, Dominance** from text, trained on the **EmoBank** corpus (~10k sentences).

```
Input text → DeBERTa-v3-base encoder → pooled [CLS] (768) → Linear(768 → 3) → [V, A, D]
```

**Before running:** `Runtime → Change runtime type → GPU` (a T4 is enough). Full training takes roughly 15–25 minutes on a T4.

## 1. Install dependencies

In [ ]:
!pip install -q -U transformers datasets accelerate evaluate scikit-learn scipy sentencepiece protobuf

## 2. Imports and setup

In [ ]:
import os, random
import numpy as np
import pandas as pd
import torch
from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    set_seed,
)
import transformers

SEED = 42
set_seed(SEED)

MODEL_NAME = "microsoft/deberta-v3-base"
OUTPUT_DIR = "./deberta_vad"
MAX_LENGTH = 128
LABEL_COLS = ["valence", "arousal", "dominance"]

print("transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None (switch runtime to GPU!)")

## 3. Load EmoBank

EmoBank is downloaded directly from its official GitHub repo (JULIELab). Columns: `id, split, V, A, D, text`. Scores are on a **1–5** scale and the corpus ships with its own `train / dev / test` split, which we keep.

In [ ]:
EMOBANK_URL = "https://raw.githubusercontent.com/JULIELab/EmoBank/master/corpus/emobank.csv"

df = pd.read_csv(EMOBANK_URL)
df = df.dropna(subset=["text"])            # one row has empty text
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 0]

print(df.shape)
print(df["split"].value_counts())
df.head()

In [ ]:
df[["V", "A", "D"]].describe()

## 4. Normalize labels to 0–1

The guide uses a 0–1 scale, so we map EmoBank's 1–5 with `(x - 1) / 4`. Predictions are converted back to 1–5 for reporting when needed.

Note: EmoBank scores cluster tightly around the neutral midpoint (std ≈ 0.2–0.35 on the 1–5 scale), so MAE will look small. **Pearson r** and **R²** are the more informative metrics here.

In [ ]:
def to_unit(x):   return (x - 1.0) / 4.0
def from_unit(x): return x * 4.0 + 1.0

df["valence"]   = to_unit(df["V"])
df["arousal"]   = to_unit(df["A"])
df["dominance"] = to_unit(df["D"])

def make_split(name):
    part = df[df["split"] == name][["text"] + LABEL_COLS].reset_index(drop=True)
    return Dataset.from_pandas(part)

raw_ds = DatasetDict({
    "train": make_split("train"),
    "validation": make_split("dev"),
    "test": make_split("test"),
})
raw_ds

## 5. Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(tokenizer("What an incredible pentakill!"))

## 6. Tokenize and build label vectors

Each sample gets `labels = [valence, arousal, dominance]`. Padding is done per batch by `DataCollatorWithPadding` (faster than padding everything to 128).

In [ ]:
def preprocess(batch):
    enc = tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)
    enc["labels"] = [
        [float(v), float(a), float(d)]
        for v, a, d in zip(batch["valence"], batch["arousal"], batch["dominance"])
    ]
    return enc

tokenized_ds = raw_ds.map(
    preprocess,
    batched=True,
    remove_columns=raw_ds["train"].column_names,
)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_ds

## 7. Regression model

`num_labels=3` + `problem_type="regression"` gives a 3-output linear head trained with MSE loss. The head is newly initialized, so the warning about uninitialized `classifier` / `pooler` weights is expected.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    problem_type="regression",
)
model.config.id2label = {i: n for i, n in enumerate(LABEL_COLS)}
model.config.label2id = {n: i for i, n in enumerate(LABEL_COLS)}

## 8. Metrics (MAE, RMSE, R², Pearson — overall and per dimension)

In [ ]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.asarray(preds, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.float64)

    out = {
        "mae":  mean_absolute_error(labels, preds),
        "rmse": float(np.sqrt(mean_squared_error(labels, preds))),
        "r2":   r2_score(labels, preds),
    }
    rs = []
    for i, name in enumerate(LABEL_COLS):
        r = pearsonr(labels[:, i], preds[:, i])[0]
        rs.append(r)
        out[f"pearson_{name}"] = float(r)
        out[f"mae_{name}"] = mean_absolute_error(labels[:, i], preds[:, i])
    out["pearson_mean"] = float(np.mean(rs))
    return out

## 9. Training configuration

Starting values from the guide (lr 2e-5, batch 16, max length 128), with a few Colab-friendly additions: fp16 on GPU, warmup, early stopping, and picking the best checkpoint by mean Pearson r.

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    eval_strategy="epoch",          # called `evaluation_strategy` in transformers < 4.41
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="pearson_mean",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",               # no wandb prompt
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

## 10. Train

In [ ]:
trainer.train()

## 11. Evaluate on the held-out test set

In [ ]:
test_metrics = trainer.evaluate(tokenized_ds["test"], metric_key_prefix="test")
pd.Series(test_metrics).round(4)

In [ ]:
# Predicted vs. true on the test set (1–5 scale)
import matplotlib.pyplot as plt

pred_out = trainer.predict(tokenized_ds["test"])
preds = from_unit(pred_out.predictions)
labels = from_unit(pred_out.label_ids)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for i, name in enumerate(LABEL_COLS):
    ax = axes[i]
    ax.scatter(labels[:, i], preds[:, i], s=6, alpha=0.4)
    lo, hi = 1, 5
    ax.plot([lo, hi], [lo, hi], "r--", lw=1)
    r = pearsonr(labels[:, i], preds[:, i])[0]
    ax.set_title(f"{name}  (r = {r:.3f})")
    ax.set_xlabel("true"); ax.set_ylabel("predicted")
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
plt.tight_layout(); plt.show()

## 12. Save the model

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
!ls -la {OUTPUT_DIR}

Colab storage is wiped when the session ends. Run this cell to copy the model to Google Drive:

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/models
!cp -r {OUTPUT_DIR} /content/drive/MyDrive/models/

## 13. Inference

In [ ]:
infer_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
infer_model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
device = "cuda" if torch.cuda.is_available() else "cpu"
infer_model.to(device).eval()

@torch.no_grad()
def predict_vad(texts, scale="unit"):
    """scale='unit' → 0–1 (guide format), scale='emobank' → original 1–5."""
    if isinstance(texts, str):
        texts = [texts]
    enc = infer_tokenizer(texts, truncation=True, max_length=MAX_LENGTH,
                          padding=True, return_tensors="pt").to(device)
    out = infer_model(**enc).logits.float().cpu().numpy()
    out = np.clip(out, 0.0, 1.0)
    if scale == "emobank":
        out = from_unit(out)
    return pd.DataFrame(out, columns=LABEL_COLS, index=texts).round(3)

predict_vad([
    "What an unbelievable teamfight!",
    "What an incredible pentakill!",
    "Blue team secures dragon.",
    "The fight is slowly developing.",
    "They threw the game, this is a disaster.",
])

## 14. (Optional) Push to the Hugging Face Hub

Needs a Hugging Face token with **write** access (https://huggingface.co/settings/tokens).

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
HUB_REPO = "your-username/deberta-v3-base-emobank-vad"   # ← change this
infer_model.push_to_hub(HUB_REPO)
infer_tokenizer.push_to_hub(HUB_REPO)